# 第 12 章：PCA 與 SVD 實務應用

本 Notebook 是 `ch12_pca_applications.md` 教學文件的精簡互動版，程式碼與 `ch12_pca_applications.py` 邏輯一致。

## 學習目標

- 理解 PCA（主成分分析）的動機：在高維資料中找出變異最大的方向，進行降維
- 能手算並用程式實作資料中心化、共變異數矩陣
- 理解「對共變異數矩陣做特徵分解」與「對資料矩陣做 SVD」是等價的兩種 PCA 做法，並能交叉驗證
- 能將資料投影到主成分方向，完成降維
- 理解 SVD 低秩近似的原理，能用前 k 個奇異值重建近似矩陣，並比較不同 k 的誤差與壓縮率

---

## Part A：主成分分析 (PCA)


In [1]:
import os

import matplotlib
matplotlib.use("Agg")  # Notebook 環境仍以檔案輸出圖片，避免依賴互動式後端

import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = [
    "PingFang TC", "PingFang SC", "Heiti TC", "Arial Unicode MS", "DejaVu Sans",
]
matplotlib.rcParams["axes.unicode_minus"] = False

OUT_DIR = os.getcwd()

np.random.seed(0)
print("環境設定完成，random seed = 0")


環境設定完成，random seed = 0


### 1. 產生一組具相關性的 2D 模擬資料

我們先在旋轉前的座標系中，讓兩個方向的標準差不同（主軸方向變異較大），再旋轉一個角度，
使得資料在 $x_1, x_2$ 兩個座標軸上呈現明顯的線性相關性。

In [2]:
n_samples = 200

theta = np.radians(30)
rotation = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta), np.cos(theta)],
])

latent = np.random.randn(n_samples, 2) * np.array([3.0, 0.8])
X = latent @ rotation.T

true_mean = np.array([5.0, 2.0])
X = X + true_mean

print("資料矩陣 X 形狀 =", X.shape)
print("X 前 5 筆樣本：")
print(X[:5])
print("X 的平均值（中心化前）=", X.mean(axis=0))


資料矩陣 X 形狀 = (200, 2)
X 前 5 筆樣本：
[[ 9.42307955  4.92331557]
 [ 6.64647859  5.02064333]
 [10.24296914  4.12425901]
 [ 7.528945    3.32026928]
 [ 4.56759016  2.12964271]]
X 的平均值（中心化前）= [4.80692612 1.90230572]


### 2. 資料中心化 (centering)

PCA 的第一步是把資料減去平均值，讓資料以原點為中心。是否要再除以標準差（標準化）取決於各變數尺度是否相近；
本例兩變數尺度接近，故只做中心化。

In [3]:
X_mean = X.mean(axis=0)
X_centered = X - X_mean

print("平均值 mean(X) =", X_mean)
print("中心化後平均值（應接近 0）=", X_centered.mean(axis=0))


平均值 mean(X) = [4.80692612 1.90230572]
中心化後平均值（應接近 0）= [-2.45581333e-15 -5.55111512e-16]


### 3. 共變異數矩陣與特徵分解

共變異數矩陣 $C = \dfrac{1}{n-1} X_c^\top X_c$，對 $C$ 做特徵分解 $C v = \lambda v$：
特徵向量即主成分方向，特徵值代表該方向上的變異量。

In [4]:
n = X_centered.shape[0]
cov_matrix = (X_centered.T @ X_centered) / (n - 1)
print("共變異數矩陣 C =")
print(cov_matrix)

cov_matrix_np = np.cov(X_centered, rowvar=False)
print("np.cov 驗證是否一致：", np.allclose(cov_matrix, cov_matrix_np))

eigvals, eigvecs = np.linalg.eigh(cov_matrix)
order = np.argsort(eigvals)[::-1]
eigvals_sorted = eigvals[order]
eigvecs_sorted = eigvecs[:, order]

pc1_eig = eigvecs_sorted[:, 0]
pc2_eig = eigvecs_sorted[:, 1]

print("\n特徵值（由大到小）=", eigvals_sorted)
print("PC1（特徵分解法）=", pc1_eig)
print("PC2（特徵分解法）=", pc2_eig)

explained_ratio_eig = eigvals_sorted / eigvals_sorted.sum()
print("各主成分解釋的變異量比例 =", explained_ratio_eig)


共變異數矩陣 C =
[[6.96965728 3.56564662]
 [3.56564662 2.61518761]]
np.cov 驗證是否一致： True

特徵值（由大到小）= [8.97024333 0.61460156]
PC1（特徵分解法）= [-0.87210701 -0.48931521]
PC2（特徵分解法）= [ 0.48931521 -0.87210701]
各主成分解釋的變異量比例 = [0.93587778 0.06412222]


### 4. 用 SVD 重做一次 PCA，並交叉驗證

對中心化資料 $X_c$ 直接做 SVD：$X_c = U \Sigma V^\top$。可以證明
$C = X_c^\top X_c / (n-1) = V \,\mathrm{diag}(\Sigma^2/(n-1))\, V^\top$，
所以 SVD 的右奇異向量 $V$ 就是共變異數矩陣的特徵向量，且 $\lambda_i = \sigma_i^2/(n-1)$。

特徵向量方向可能相差正負號，比較前需先用內積正負號對齊方向。

In [5]:
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
pc_svd = Vt.T
eigvals_from_svd = (S ** 2) / (n - 1)

print("SVD 奇異值 S =", S)
print("由奇異值換算出的特徵值 =", eigvals_from_svd)
print("特徵分解法特徵值           =", eigvals_sorted)
print("兩者是否一致：", np.allclose(eigvals_from_svd, eigvals_sorted))


def align_sign(v_ref, v_target):
    """用內積正負號對齊 v_target 與 v_ref 的方向（特徵向量方向不唯一，可能差一個負號）"""
    if np.dot(v_ref, v_target) < 0:
        return -v_target
    return v_target


pc1_svd_aligned = align_sign(pc1_eig, pc_svd[:, 0])
pc2_svd_aligned = align_sign(pc2_eig, pc_svd[:, 1])

pc1_match = np.allclose(pc1_eig, pc1_svd_aligned)
pc2_match = np.allclose(pc2_eig, pc2_svd_aligned)
print("\nPC1 校正正負號後是否一致：", pc1_match)
print("PC2 校正正負號後是否一致：", pc2_match)
print(">>> 驗證通過" if (pc1_match and pc2_match) else ">>> 驗證失敗")


SVD 奇異值 S = [42.25018844 11.05919123]
由奇異值換算出的特徵值 = [8.97024333 0.61460156]
特徵分解法特徵值           = [8.97024333 0.61460156]
兩者是否一致： True

PC1 校正正負號後是否一致： True
PC2 校正正負號後是否一致： True
>>> 驗證通過


### 5. 投影到第一主成分：資料降維

把中心化資料投影到 PC1 方向：$z = X_c \, v_1$，即可把 2D 資料降成 1D，同時保留最多變異量。

In [6]:
z_1d = X_centered @ pc1_eig
print("投影到 PC1 後的 1D 座標（前 5 筆）=", z_1d[:5])

X_reconstructed_1pc = np.outer(z_1d, pc1_eig) + X_mean
reconstruction_error = np.linalg.norm(X - X_reconstructed_1pc, "fro")
print("只用第一主成分重建的 Frobenius 誤差 =", reconstruction_error)
print("第一主成分解釋的變異量比例 =", explained_ratio_eig[0])


投影到 PC1 後的 1D 座標（前 5 筆）= [-5.5040058  -3.13013661 -5.82804673 -3.06772286  0.09748713]
只用第一主成分重建的 Frobenius 誤差 = 11.059191229296285
第一主成分解釋的變異量比例 = 0.9358777770658615


### 6. 繪圖：原始資料點、主成分方向與降維結果

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], s=15, alpha=0.5, color="tab:blue", label="原始資料點")

scale = 2.0
pc1_end = X_mean + scale * np.sqrt(eigvals_sorted[0]) * pc1_eig
pc2_end = X_mean + scale * np.sqrt(eigvals_sorted[1]) * pc2_eig

ax.annotate("", xy=pc1_end, xytext=X_mean,
            arrowprops=dict(arrowstyle="->", color="tab:red", linewidth=2.5))
ax.annotate("", xy=pc2_end, xytext=X_mean,
            arrowprops=dict(arrowstyle="->", color="tab:green", linewidth=2.5))
ax.plot([], [], color="tab:red", linewidth=2.5, label="PC1")
ax.plot([], [], color="tab:green", linewidth=2.5, label="PC2")
ax.scatter(*X_mean, color="black", marker="x", s=80, label="資料平均值")

ax.set_title("原始資料點與主成分方向")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.axis("equal")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="best", fontsize=9)

ax2 = axes[1]
ax2.scatter(z_1d, np.zeros_like(z_1d), s=15, alpha=0.5, color="tab:red")
ax2.axhline(0, color="gray", linewidth=1)
ax2.set_title("投影到第一主成分後的 1D 降維結果")
ax2.set_xlabel("PC1 座標 z")
ax2.set_yticks([])
ax2.grid(True, linestyle="--", alpha=0.5, axis="x")

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "pca_projection.png")
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("已儲存 PCA 圖示至:", fig_path)


已儲存 PCA 圖示至: /Users/rexwang/workspace/linear-algebra-with-matlab-python-tutorial/ch12_pca_applications/pca_projection.png


/var/folders/3r/mbt455ns6n3fn8ythcyc95qc0000gn/T/ipykernel_53339/903949967.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


![PCA projection](pca_projection.png)

---

## Part B：SVD 影像/矩陣壓縮

### 1. 用外積組合出一個有結構的簡單「圖像」矩陣

用幾個 rank-1 外積成分疊加出條紋與漸層圖案，再加一點微小雜訊，避免依賴外部圖檔。

In [8]:
size = 100
coords = np.linspace(-3, 3, size)

pattern_1 = np.outer(np.sin(coords), np.cos(coords))
pattern_2 = np.outer(np.exp(-coords ** 2 / 4), np.ones(size))
pattern_3 = np.outer(np.ones(size), np.linspace(-1, 1, size))
pattern_4 = np.outer(np.cos(2 * coords), np.sin(3 * coords))

rng = np.random.RandomState(42)
noise = 0.05 * rng.randn(size, size)

image = 3.0 * pattern_1 + 2.0 * pattern_2 + 1.0 * pattern_3 + 0.5 * pattern_4 + noise

print("圖像矩陣形狀 =", image.shape)
print("圖像矩陣的秩 =", np.linalg.matrix_rank(image))


圖像矩陣形狀 = (100, 100)
圖像矩陣的秩 = 100


### 2. 對圖像矩陣做 SVD

In [9]:
U_img, S_img, Vt_img = np.linalg.svd(image, full_matrices=False)
print("U 形狀 =", U_img.shape, "; S 形狀 =", S_img.shape, "; Vt 形狀 =", Vt_img.shape)
print("前 10 個奇異值 =", S_img[:10])


U 形狀 = (100, 100) ; S 形狀 = (100,) ; Vt 形狀 = (100, 100)
前 10 個奇異值 = [150.89714024 137.64940707  26.47535661  23.00664966   0.97907918
   0.93676035   0.9127278    0.89020155   0.88276139   0.85168628]


### 3. 低秩近似重建：比較不同 k 值的誤差與壓縮率

低秩近似：$A_k = U_k \Sigma_k V_k^\top$，只用前 k 個奇異值/奇異向量重建矩陣。

In [10]:
def low_rank_approx(U, S, Vt, k):
    return U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]


k_values = [1, 2, 3, 5, 10, 20, 50]
m, n_cols = image.shape
full_rank = min(m, n_cols)

errors = []
compression_ratios = []

print(f"{'k':>4} | {'Frobenius 誤差':>16} | {'相對誤差':>10} | {'壓縮率':>10}")
print("-" * 52)
for k in k_values:
    A_k = low_rank_approx(U_img, S_img, Vt_img, k)
    err = np.linalg.norm(image - A_k, "fro")
    rel_err = err / np.linalg.norm(image, "fro")
    storage_k = k * (m + n_cols + 1)
    storage_full = m * n_cols
    compression_ratio = storage_k / storage_full

    errors.append(err)
    compression_ratios.append(compression_ratio)
    print(f"{k:>4} | {err:>16.6f} | {rel_err:>10.4%} | {compression_ratio:>10.4%}")

errors_arr = np.array(errors)
is_monotonic_decreasing = np.all(np.diff(errors_arr) <= 1e-10)
print()
print(">>> 驗證通過：誤差隨 k 增加而單調遞減。" if is_monotonic_decreasing else ">>> 驗證失敗")

A_full = low_rank_approx(U_img, S_img, Vt_img, full_rank)
full_rank_error = np.linalg.norm(image - A_full, "fro")
print(f"k = full_rank ({full_rank}) 時的重建誤差 = {full_rank_error:.2e}（應接近 0）")


   k |     Frobenius 誤差 |       相對誤差 |        壓縮率
----------------------------------------------------
   1 |       142.130243 |   68.5645% |    2.0100%
   2 |        35.406872 |   17.0805% |    4.0200%
   3 |        23.509616 |   11.3412% |    6.0300%
   5 |         4.736828 |    2.2851% |   10.0500%
  10 |         4.293000 |    2.0710% |   20.1000%
  20 |         3.511636 |    1.6940% |   40.2000%
  50 |         1.668205 |    0.8048% |  100.5000%

>>> 驗證通過：誤差隨 k 增加而單調遞減。
k = full_rank (100) 時的重建誤差 = 1.51e-13（應接近 0）


### 4. 繪圖：原始矩陣 vs 不同 k 值的重建結果

In [11]:
k_to_plot = [1, 3, 10, 50]
vmin, vmax = image.min(), image.max()

fig, axes = plt.subplots(1, len(k_to_plot) + 1, figsize=(4 * (len(k_to_plot) + 1), 4.2))

im0 = axes[0].imshow(image, cmap="viridis", vmin=vmin, vmax=vmax)
axes[0].set_title("原始矩陣（滿秩）")
axes[0].axis("off")

for ax, k in zip(axes[1:], k_to_plot):
    A_k = low_rank_approx(U_img, S_img, Vt_img, k)
    err = np.linalg.norm(image - A_k, "fro")
    ax.imshow(A_k, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"k = {k}\n誤差 = {err:.3f}")
    ax.axis("off")

fig.colorbar(im0, ax=axes, fraction=0.02, pad=0.02)
fig_path2 = os.path.join(OUT_DIR, "svd_compression.png")
plt.savefig(fig_path2, dpi=120, bbox_inches="tight")
plt.show()
print("已儲存 SVD 低秩近似比較圖至:", fig_path2)


已儲存 SVD 低秩近似比較圖至: /Users/rexwang/workspace/linear-algebra-with-matlab-python-tutorial/ch12_pca_applications/svd_compression.png


/var/folders/3r/mbt455ns6n3fn8ythcyc95qc0000gn/T/ipykernel_53339/907954826.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(k_values, errors, marker="o", color="tab:purple")
ax.set_title("重建誤差（Frobenius norm）隨 k 增加而遞減")
ax.set_xlabel("保留的奇異值個數 k")
ax.set_ylabel("Frobenius 誤差 ||A - A_k||_F")
ax.grid(True, linestyle="--", alpha=0.5)
fig_path3 = os.path.join(OUT_DIR, "svd_error_curve.png")
plt.savefig(fig_path3, dpi=120, bbox_inches="tight")
plt.show()
print("已儲存誤差遞減曲線圖至:", fig_path3)


已儲存誤差遞減曲線圖至: /Users/rexwang/workspace/linear-algebra-with-matlab-python-tutorial/ch12_pca_applications/svd_error_curve.png


/var/folders/3r/mbt455ns6n3fn8ythcyc95qc0000gn/T/ipykernel_53339/2129147719.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


![SVD compression](svd_compression.png)

![SVD error curve](svd_error_curve.png)

---

## 小結

本章完整走過「PCA 兩種等價實作方式（特徵分解 vs SVD）」與「SVD 低秩近似壓縮」，
兩者都建立在第 8 章（特徵值）與第 10 章（SVD）的基礎上。更完整的說明、公式推導與練習題請見 `ch12_pca_applications.md`。